# Label Audit: Which Boiling Points Are Real Measurements?

In the rich-features study (`Boiling_Point_RDKit.ipynb`), about 20 compounds decided every RMSE comparison. Every model and feature set missed them badly, often by hundreds of kelvin, in the same direction. When models with very different assumptions all disagree with a label the same way, the label itself becomes a suspect.

This notebook audits all 1,588 literature labels with three checks. **None of them uses a model's predictions as evidence.** Consistent model disagreement only nominates compounds for an independent lookup. It then repeats the nested-CV study on the labels that survive.

Code: `src/boiling_point/audit.py` and `scripts/audit_labels.py` (writes `data/label_audit.csv`); the re-run is `scripts/run_feature_study.py --audited`.

In [1]:
import sys
sys.path.insert(0, "src")
sys.path.insert(0, "scripts")

from boiling_point import audit, data, features  # boiling_point first: loads xgboost before torch

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

literature = data.build_modelling_table(
    data.load_literature_data("data/compound_boiling_points_from_literature.xlsx"), data.load_pubchem_data())
label_audit = pd.read_csv("data/label_audit.csv")
print(f"{len(literature)} literature compounds")

1588 literature compounds


## Check 1: labels that are Joback estimates

Joback & Reid's **group-contribution method** estimates a normal boiling point as 198.2 K plus a fixed amount for every structural group: −CH₃ adds 23.58 K, −CH₂− 22.88 K, an alcohol −OH 92.88 K, and so on. For 3-methyltritriacontane (C₃₄H₇₀: 3 CH₃, 30 CH₂, 1 CH) that gives 198.2 + 70.74 + 686.40 + 21.74 = **977.08 K**. That is exactly its label, to the hundredth of a kelvin.

A measured value will rarely land within 0.005 K of this formula by chance: Joback typically misses real boiling points by ~10–20 K. So a label *identical* to the estimate was almost certainly computed, not measured. The table counts how many labels fall at each distance from their Joback estimate (for the compounds whose groups are all covered):

In [2]:
joback = audit.joback_matches(literature)
distance = (joback["boiling_point_kelvin"] - joback["joback_tb"]).abs().dropna()
bands = [(0, 0.005), (0.005, 0.015), (0.015, 0.1), (0.1, 0.5), (0.5, 1), (1, 2)]
background = ((distance >= 0.1) & (distance < 2)).sum() / 1.9    # labels per K, away from zero
pd.DataFrame([{"|label − Joback| (K)": f"{lo}–{hi}", "labels": int(((distance >= lo) & (distance < hi)).sum()),
               "expected by chance": round(background * 2 * (hi - lo) / 2, 1)} for lo, hi in bands])

,|label − Joback| (K),labels,expected by chance
0,0–0.005,327,0.3
1,0.005–0.015,2,0.5
2,0.015–0.1,7,4.5
3,0.1–0.5,27,21.3
4,0.5–1,24,26.6
5,1–2,50,53.2


**327 labels are identical to their Joback estimate, where fewer than one (about 0.3) would be expected by chance.** They span the whole range, from small unsaturated hydrocarbons around 287 K to the C₅₀ alkane at 1,343 K. The two labels just 0.01 K away (3-methylpentane, 1-butyne) are well-known compounds whose measured values happen to lie next to the estimate. They are treated as measurements.

How wrong are the Joback labels likely to be? Comparing the formula with the *measured* labels shows it's decent for small molecules but increasingly too high for large ones:

In [3]:
excluded = set(label_audit.loc[label_audit["exclude"], "cmpdname"])
measured = joback[~joback["cmpdname"].isin(excluded) & joback["joback_tb"].notna()].join(literature[["heavycnt"]])
measured["Joback − measured (K)"] = measured["joback_tb"] - measured["boiling_point_kelvin"]
size = pd.cut(measured["heavycnt"], [0, 6, 10, 15, 20, 30, 60], labels=["≤6", "7–10", "11–15", "16–20", "21–30", ">30"])
table = measured.groupby(size, observed=True)["Joback − measured (K)"].agg(
    compounds="size", median_bias="median", mean_abs_error=lambda e: e.abs().mean()).round(1)
table.index.name = "heavy atoms"
jb = joback[joback["label_is_joback"]].join(literature[["heavycnt"]])
print(f"Joback-labelled compounds with more than 20 heavy atoms: {(jb['heavycnt'] > 20).sum()} of {len(jb)}")
table

Joback-labelled compounds with more than 20 heavy atoms: 58 of 327


,compounds,median_bias,mean_abs_error
heavy atoms,,,
≤6,139,3.2,14.5
7–10,476,0.9,19.2
11–15,283,0.5,15.8
16–20,50,38.1,51.8
21–30,31,109.0,128.0
>30,17,238.2,258.1


For molecules with more than ~15 heavy atoms, Joback increasingly overshoots, by roughly 100–250 K beyond 20 heavy atoms. So the Joback labels on large molecules are probably far too high. That's exactly where the models kept "underpredicting": they were closer to the truth than the labels.

## Check 2: physically implausible hydrocarbons

An acyclic hydrocarbon boils close to the straight-chain alkane with the same number of carbons. Even heavy branching lowers the boiling point only ~20–30 K (neopentane 283 K vs pentane 309 K). The check compares every hydrocarbon label, not just model-flagged ones, with the measured n-alkane boiling point (CRC Handbook, C₁–C₂₄; longer alkanes decompose before boiling):

In [4]:
carbons = literature["isosmiles"].map(audit.hydrocarbon_carbon_count)
hc = literature[carbons.notna() & carbons.le(24)].assign(n_carbons=carbons)
below = hc["n_carbons"].map(audit.N_ALKANE_TB) - hc["boiling_point_kelvin"]
print(f"{len(hc)} hydrocarbons: label below the matching n-alkane by "
      f"median {below.median():.0f} K, 99th percentile {below.quantile(0.99):.0f} K")
audit.implausible_hydrocarbons(literature)[["cmpdname", "n_carbons", "boiling_point_kelvin", "n_alkane_tb", "below_n_alkane"]].round(1)

842 hydrocarbons: label below the matching n-alkane by median 10 K, 99th percentile 40 K


,cmpdname,n_carbons,boiling_point_kelvin,n_alkane_tb,below_n_alkane
269,Phytane,20.0,442.6,616.9,174.2
640,3-Decyne,10.0,320.3,447.3,127.0
919,4-Nonyne,9.0,337.8,424.0,86.2
941,9-Octadecyne,18.0,415.0,589.9,174.9
1000,2-Methyltricosane,24.0,480.9,664.5,183.6
1271,beta-Ocimene,10.0,346.2,447.3,101.2
1277,"3,7-Dimethylocta-1,3,7-triene",10.0,354.2,447.3,93.2


99% of hydrocarbons sit within ~40 K of the matching n-alkane. Seven labels fall more than 80 K below it: a C₁₈ alkyne at 415 K, a C₂₄ alkane at 481 K. Such values look like boiling points measured at **reduced pressure** and recorded as normal boiling points.

## Check 3: independent values for compounds every model mispredicts

The compounds that every out-of-fold prediction misses by more than 50 K in the same direction (48 predictions each: 4 feature sets × 4 models × 3 repeats) were looked up on the NIST Chemistry WebBook. The model disagreement only *selects* what to look up; the evidence is the independent value.

In [5]:
nist = pd.read_csv("data/nist_audit_lookups.csv")
nist["prediction"] = nist["label"] + nist["mean_residual"]
found = nist[nist["nist_status"] == "found"][["cmpdname", "label", "nist_tb", "prediction"]]
print(f"{len(nist)} compounds looked up; NIST has a boiling point for {len(found)}")
found.round(1)

30 compounds looked up; NIST has a boiling point for 5


,cmpdname,label,nist_tb,prediction
10,3-Decyne,320.3,449.0,443.3
19,4-Nonyne,337.8,425.0,417.8
20,9-Octadecyne,415.0,415.0,605.0
22,"3,9-Diethyl-6-tridecanol",582.2,582.2,659.2
25,"N,N-diethylhexadecylamine",472.6,628.2,617.6


- **Three labels are contradicted by NIST:** 4-nonyne, 3-decyne and N,N-diethylhexadecylamine. They are 87–155 K too low, and NIST agrees with the *model predictions* to within ~10 K.
- **3,9-Diethyl-6-tridecanol is confirmed.** There the label is right and the models are wrong.
- **9-Octadecyne matches NIST,** but at 415 K it also fails check 2. Both probably trace to one reduced-pressure value.
- **The rest have no NIST boiling point,** so they stay in the data. Removing compounds only because models disagree with them would be circular.

## Audit result

In [6]:
summary = label_audit.groupby(["category", "exclude"]).size().rename("compounds").reset_index()
print(f"excluded: {label_audit['exclude'].sum()} of {len(literature)} labels -> "
      f"{len(literature) - label_audit['exclude'].sum()} measured labels")
summary

excluded: 337 of 1588 labels -> 1251 measured labels


,category,exclude,compounds
0,Joback estimate,True,324
1,"Joback estimate; model-flagged, unverified",True,3
2,NIST agrees,False,1
3,contradicted by NIST,True,1
4,implausible hydrocarbon,True,1
5,implausible hydrocarbon; NIST agrees,True,1
6,implausible hydrocarbon; contradicted by NIST,True,2
7,"implausible hydrocarbon; model-flagged, unveri...",True,3
8,known data-entry error,True,2
9,"model-flagged, unverified",False,19


In [7]:
from boiling_point import families
fam = families.assign_families(literature["isosmiles"])
kept = ~literature["cmpdname"].isin(excluded)
pd.DataFrame({"all labels": fam.value_counts(), "measured labels": fam[kept.to_numpy()].value_counts()})

,all labels,measured labels
family,,
alcohol/phenol,526,352
alkane,495,388
alkene/alkyne,405,366
amine,159,142
other,3,3


The exclusions fall unevenly across families: a third of the alcohol labels are Joback estimates.

## Re-evaluation on measured labels

Same protocol as the rich-features study (nested CV, 5 folds × 3 repeats, stratified by family, tuned by MAE), on the 1,251 measured labels. First, the paired comparison with the original 12 features on the same folds:

In [8]:
metrics = pd.read_csv("results/feature_study_audited/fold_metrics.csv")
MODELS = ["ridge", "xgboost", "mlp", "ensemble"]
rows = []
for metric in ("mae", "rmse"):
    wide = metrics[metrics["model"].isin(MODELS)].pivot_table(index=["model", "repeat", "fold"],
                                                              columns="feature_set", values=metric)
    for fs in ["curated", "curated (log)"]:
        for model, d in (wide[fs] - wide["old 12"]).groupby("model"):
            rows.append({"metric": metric.upper(), "feature set": fs, "model": model,
                         "mean change vs old 12 (K)": round(d.mean(), 1),
                         "folds better than old 12": f"{(d < 0).sum()}/{len(d)}"})
pd.DataFrame(rows).set_index(["metric", "feature set", "model"])

mean change vs old 12 (K)  \
metric feature set   model                                 
MAE    curated       ensemble                       -2.2   
                     mlp                            -0.9   
                     ridge                          -5.7   
                     xgboost                        -0.6   
       curated (log) ensemble                       -2.1   
                     mlp                             0.7   
                     ridge                          -6.6   
                     xgboost                        -0.6   
RMSE   curated       ensemble                       -2.0   
                     mlp                            -1.0   
                     ridge                          -6.2   
                     xgboost                        -0.5   
       curated (log) ensemble                       -2.4   
                     mlp                            -0.7   
                     ridge                          -7.6   
                     xgboost                        -0.5   

                              folds better than old 12  
metric feature set   model                              
MAE    curated       ensemble                    15/15  
                     mlp                          9/15  
                     ridge                       15/15  
                     xgboost                     12/15  
       curated (log) ensemble                    14/15  
                     mlp                          6/15  
                     ridge                       15/15  
                     xgboost                     12/15  
RMSE   curated       ensemble                    12/15  
                     mlp                          6/15  
                     ridge                       15/15  
                     xgboost                     10/15  
       curated (log) ensemble                    13/15  
                     mlp                          8/15  
                     ridge                       15/15  
                     xgboost                     10/15

In [9]:
(metrics[metrics["model"].isin(MODELS)].groupby(["feature_set", "model"])[["mae", "rmse", "rmse_hard"]]
 .agg(["mean", "std"]).round(1))

mae       rmse      rmse_hard      
                                 mean  std  mean  std      mean   std
feature_set            model                                         
curated                ensemble  10.8  0.9  22.5  4.7      48.7  15.0
                       mlp       11.9  1.7  24.0  4.8      48.2  14.2
                       ridge     15.2  0.8  26.4  4.2      53.8  14.5
                       xgboost   10.7  1.3  24.7  5.9      56.2  16.9
curated (log)          ensemble  10.9  1.1  22.1  4.4      48.5  13.4
                       mlp       13.5  3.6  24.3  4.4      47.3  10.9
                       ridge     14.4  0.6  24.9  4.1      52.5  13.8
                       xgboost   10.7  1.3  24.7  5.9      56.2  16.9
curated (log) + family ensemble  10.7  1.2  22.6  5.1      50.2  15.9
                       mlp       13.3  4.5  25.8  7.1      54.1  18.9
                       ridge     13.5  0.7  24.3  3.6      51.5  11.7
                       xgboost   10.8  1.3  24.9  5.9      56.5  17.3
old 12                 ensemble  12.9  1.4  24.5  5.6      54.3  17.2
                       mlp       12.8  2.4  25.0  7.0      53.4  20.8
                       ridge     21.0  1.1  32.5  4.2      64.3  15.9
                       xgboost   11.3  1.3  25.2  6.1      59.6  17.5

**On measured labels, the curated descriptors win on both metrics:**
- the ensemble is 2.2 K better by MAE, in every one of the 15 folds, and 2.0 K better by RMSE;
- Ridge gains most (~6 K);
- XGBoost and the MLP gain ~0.5–1 K.

The best recipes reach **~10.7 K MAE and ~22 K RMSE**. Hard-region RMSE drops from ~70 K to ~48 K. With the misleading labels gone, the physically motivated descriptors finally show their value.

The log transform is now roughly neutral for the ensemble, helps Ridge and hurts the MLP.

## Does cleaner training data help, or were the removed compounds just the hard ones?

Both. The audited run is scored on different compounds from the full run, and the removed ones include the hardest giant molecules. To separate the two effects, this compares both runs **on the same 1,251 measured compounds**. It uses each run's out-of-fold predictions: one set of models trained with the Joback and other suspect labels included, the other without them.

In [10]:
full = pd.read_csv("results/feature_study/predictions.csv").merge(
    pd.read_csv("results/feature_study/compounds.csv")[["row", "cmpdname"]], on="row")
aud = pd.read_csv("results/feature_study_audited/predictions.csv").merge(
    pd.read_csv("results/feature_study_audited/compounds.csv")[["row", "cmpdname"]], on="row")
measured_names = set(aud["cmpdname"])
rows = []
for fs in ["old 12", "curated", "curated (log)"]:
    for model in MODELS:
        f = full[(full["feature_set"] == fs) & (full["model"] == model) & full["cmpdname"].isin(measured_names)]
        a = aud[(aud["feature_set"] == fs) & (aud["model"] == model)]
        ef, ea = f["y_pred"] - f["y_true"], a["y_pred"] - a["y_true"]
        rows.append({"feature set": fs, "model": model,
                     "MAE: trained on all labels": ef.abs().mean(), "MAE: trained on measured only": ea.abs().mean(),
                     "RMSE: all labels": np.sqrt((ef ** 2).mean()), "RMSE: measured only": np.sqrt((ea ** 2).mean())})
pd.DataFrame(rows).round(2)

,feature set,model,MAE: trained on all labels,MAE: trained on measured only,RMSE: all labels,RMSE: measured only
0,old 12,ridge,20.72,20.96,37.39,32.78
1,old 12,xgboost,14.15,11.31,29.43,25.87
2,old 12,mlp,13.98,12.75,28.84,25.90
3,old 12,ensemble,14.60,12.94,29.00,25.11
4,curated,ridge,18.83,15.23,34.19,26.67
5,curated,xgboost,13.65,10.71,31.38,25.34
6,curated,mlp,14.27,11.86,31.69,24.43
7,curated,ensemble,13.98,10.78,29.73,22.98
8,curated (log),ridge,16.96,14.41,32.88,25.26
9,curated (log),xgboost,13.65,10.71,31.38,25.34


On identical compounds, training without the suspect labels cuts the curated ensemble's MAE from ~14.0 K to ~10.8 K, and RMSE from ~30 K to ~23 K. That's despite each training fold having ~20% fewer compounds.

So the curated ensemble's improvement from 16.9 K (all compounds, full-data run) to 10.8 K splits roughly into:
- **~2.9 K from easier test compounds:** 16.9 → 14.0 K, the same models scored only on the measured compounds;
- **~3.2 K from cleaner training labels:** 14.0 → 10.8 K.

**The Joback estimates weren't just noisy; they actively taught the models to overpredict large molecules.**

## Conclusions

1. **21% of the literature labels (327 of 1,588) are Joback group-contribution estimates, not measurements.** They're identical to the formula to 0.01 K, where fewer than one such match would occur by chance. For large molecules these estimates run 100–250 K too high.
2. **A few more labels are wrong or implausible,** most likely values measured at reduced pressure: 7 hydrocarbons far below the matching n-alkane, 3 contradicted by NIST (two of them also implausible hydrocarbons), and the 2 errors found earlier. 337 labels are excluded in total; 1,251 measured labels remain.
3. **On measured labels the curated RDKit descriptors win clearly.** The ensemble is 2.2 K better by MAE in 15/15 folds, reaching **~10.7 K MAE (~22 K RMSE)**. The earlier "the old features win on RMSE" result was an artefact of the inflated labels.
4. **Cleaner training data helps on its own:** about 3 K lower MAE on the same compounds, even with ~20% less training data.
5. **Lesson: audit labels before comparing models.** The biggest improvement in this project came from finding which labels were real, not from better features or models. Consistent model disagreement is a good way to choose *what* to check, but the evidence has to come from somewhere else.